In [3]:
import pandas as pd
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260729_중소상공인지원과\\"

data = pd.read_csv(r"D:\\02_사업관리\\자체분석\\01_생활인구\kt_d_living_tme_p2_202607.csv", 
                               sep="|", dtype={'std_ymd': str}, encoding="utf-8", low_memory=False)
grid100_pg = gpd.read_file(r"D:\\02_사업관리\\자체분석\\04_공간DB\\griddata\\ngii_100m_busan.shp", crs=5179)
grid100_pg['x'] = grid100_pg.geometry.centroid.x.round(3)
grid100_pg['y'] = grid100_pg.geometry.centroid.y.round(3)
grid100_pt = gpd.GeoDataFrame(grid100_pg, geometry=gpd.points_from_xy(grid100_pg.x, grid100_pg.y))
grid100_mrkt = gpd.read_file(DATA_PATH + r"01_신청내용\\(동남지방데이터청)골목상권25개\\골목상권25개.shp", crs=5179)
# weekday_map = {'월요일': '월', '화요일': '화', '수요일': '수', '목요일': '목', '금요일': '금', '토요일': '토', '일요일': '일'}
weekday_order = ['월', '화', '수', '목', '금', '토', '일']

def aggregate_daynm2(df, pt2, grid):
    print("상권별 요일별 생활인구 집계 테스트 시작")
    # cols = [col for col in df.columns if '_t_' in col]
    # df[cols] = df[cols].astype(float)
    df['year'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:4] + "년"
    # df['weekday'] = df['std_ymd'].astype('datetime64[ns]').dt.day_name('ko_KR').map(weekday_map)
    df = pd.merge(pt2, df, left_on="gid", right_on="id", how="left")
    gdf = gpd.GeoDataFrame(df, geometry="geometry")
    sjoined = gpd.sjoin(grid, gdf, how="inner", predicate="intersects")
    sjoined['weekday'] = sjoined['std_ymd'].astype('datetime64[ns]').dt.dayofweek.map(dict(enumerate(weekday_order)))
    
    times_dict = {
        '05~08시': [col for col in sjoined.columns if col.endswith(tuple(f"_{h:02d}" for h in range(5, 9)))],
        '09~11시': [col for col in sjoined.columns if col.endswith(tuple(f"_{h:02d}" for h in range(9, 12)))],
        '12~14시': [col for col in sjoined.columns if col.endswith(tuple(f"_{h:02d}" for h in range(12, 15)))],
        '15~17시': [col for col in sjoined.columns if col.endswith(tuple(f"_{h:02d}" for h in range(15, 18)))],
        '18~22시': [col for col in sjoined.columns if col.endswith(tuple(f"_{h:02d}" for h in range(18, 23)))],
        '23~04시': [col for col in sjoined.columns if col.endswith(("_23", "_00", "_01", "_02", "_03", "_04"))],}
    for key, value in times_dict.items():
        sjoined[key] = sjoined.loc[:, value].sum(axis=1)
    sjoined['일계'] = sjoined[list(times_dict.keys())].sum(axis=1)   
    pivoted = sjoined.pivot_table(
        index=['saup_yr','상권명','year'], columns='weekday', values='일계', aggfunc='sum')
    pivoted = pivoted.reindex (columns=weekday_order).reset_index()
    pivoted.columns.name = None
    print("상권별 요일별 생활인구 집계 테스트 완료")
    return pivoted


In [4]:
aggregate_daynm2(data, grid100_pt, grid100_mrkt)

상권별 요일별 생활인구 집계 테스트 시작
상권별 요일별 생활인구 집계 테스트 완료


,saup_yr,상권명,year,월,화,수,목,금,토,일
0,2022,감천문화마을,2026년,45713.37,46334.38,58117.34,58028.79,58799.06,48482.40,49588.91
1,2022,구남야시장골목,2026년,118781.71,118093.24,148131.09,147094.51,146927.27,119666.19,124781.75
2,2022,남천해변시장,2026년,143167.12,144066.53,183488.96,180013.35,180973.40,149932.51,145426.55
3,2022,다비치상인공동체,2026년,24230.01,24178.96,30021.07,30290.47,30221.27,24865.30,24922.22
4,2022,만덕이로상권,2026년,69737.47,68877.06,86432.86,86266.35,86555.38,71695.66,74028.90
5,2022,송정서핑거리,2026년,277489.54,278822.40,342272.58,354604.24,380534.53,342900.17,335544.57
6,2022,수비 벡스코,2026년,31887.80,32679.51,41128.49,40690.50,41782.00,34411.70,32710.20
7,2022,연동되는 골목,2026년,147839.16,146087.62,182392.85,182498.01,181975.09,147652.53,150967.55
8,2022,의리단길,2026년,183416.03,180863.89,227124.55,225135.14,223963.37,178903.46,184299.87
9,2022,화승골목상권,2026년,62882.84,62715.91,78749.56,78273.71,78490.11,64865.83,66579.99


In [2]:
import pandas as pd

DATA_PATH = r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260729_중소상공인지원과\\02_집계결과\\"

gender1 = pd.read_csv(DATA_PATH + "사업연도_상권별_성별_생활인구_현황1.csv", sep="|", encoding="utf-8")
gender2 = pd.read_csv(DATA_PATH + "사업연도_상권별_성별_생활인구_현황2.csv", sep="|", encoding="utf-8")

ages1 = pd.read_csv(DATA_PATH + "사업연도_상권별_연령_생활인구_현황1.csv", sep="|", encoding="utf-8")
ages2 = pd.read_csv(DATA_PATH + "사업연도_상권별_연령_생활인구_현황2.csv", sep="|", encoding="utf-8")

daynm1 = pd.read_csv(DATA_PATH + "사업연도_상권별_요일_생활인구_현황1.csv", sep="|", encoding="utf-8")
daynm2 = pd.read_csv(DATA_PATH + "사업연도_상권별_요일_생활인구_현황2.csv", sep="|", encoding="utf-8")

times1 = pd.read_csv(DATA_PATH + "사업연도_상권별_시간_생활인구_현황1.csv", sep="|", encoding="utf-8")
times2 = pd.read_csv(DATA_PATH + "사업연도_상권별_시간_생활인구_현황2.csv", sep="|", encoding="utf-8")

pd.concat([gender1, gender2], ignore_index=True).sort_values(['사업연도','기준연도'], ascending=True)

# pd.concat([gender1, gender2], ignore_index=True).sort_values(['사업연도','기준연도'], ascending=True).to_csv(DATA_PATH + "사업연도_상권별_성별_생활인구_현황.csv", sep="|", encoding="utf-8", index=False)
# pd.concat([ages1, ages2], ignore_index=True).sort_values(['사업연도','기준연도'], ascending=True).to_csv(DATA_PATH + "사업연도_상권별_성별_생활인구_현황.csv", sep="|", encoding="utf-8", index=False)
# pd.concat([daynm1, daynm2], ignore_index=True).sort_values(['사업연도','기준연도'], ascending=True).to_csv(DATA_PATH + "사업연도_상권별_성별_생활인구_현황.csv", sep="|", encoding="utf-8", index=False)
# pd.concat([times1, times2], ignore_index=True).sort_values(['사업연도','기준연도'], ascending=True).to_csv(DATA_PATH + "사업연도_상권별_성별_생활인구_현황.csv", sep="|", encoding="utf-8", index=False)

,사업연도,상권명,기준연도,남성,여성
0,2022,감천문화마을,2021년,12087.0,12039.0
1,2022,만덕이로상권,2021년,10424.0,11787.0
2,2022,다비치상인공동체,2021년,4029.0,4086.0
3,2022,남천해변시장,2021년,23484.0,27194.0
4,2022,구남야시장골목,2021년,20777.0,20946.0
...,...,...,...,...,...
1459,2025,장림산업용품길,2025년,8090.0,7188.0
1460,2025,40계단상인회,2025년,12146.0,12762.0
1461,2025,한새벌상인회,2025년,13252.0,14442.0
1464,2025,한새벌상인회,2025년,14139.0,15159.0
